# Quantization với Pytorch

## I. Quantization là gì?

![](image1.png)

$Quantization$ là phương pháp hữu hiệu giúp tăng tốc thời gian xử lý của các mô hình học sâu mà đảm bảo độ chính xác không giảm đi đáng kể cách tính toán và lưu trữ tensor ở kiểu dữ liệu có số bit thấp hơn kiểu dữ liệu float.

Như mọi người cũng biết, mô hình học sâu chính là các phép tính toán ma trận. Ma trận được biểu diễn bằng vô vàn số đơn lẻ dưới dạng cột và hàng. Do đó độ chính xác và thời gian tính toán của mô hình học sâu liên quan mật thiết đến cách ta lưu trữ, biểu diễn các số đơn lẻ này. Trong quá trình lưu trữ các số đơn lẻ, có hai vấn đề mà ta thường gặp đó là:
- Độ chính xác (Số các số sau dấu phẩy mà chúng ta có thể lưu trữ)
- Số bit cần để biểu diễn số đó

Ví dụ một số được biểu diễn bằng int32 thì 1 bit đầu tiên dùng để biểu diễn dấu, 31 bit còn lại để biểu diễn số đó. Điều đó có nghĩa là 1 số ở dạng int32 có thể biểu diễn giá trị từ $-2^{31}$ cho đến $2^{31}$. Tương tự như vậy, một số int8 chỉ biểu diễn các giá trị từ $-2^{7}$ cho đến $2^{7}$.

Số bit càng lớn thì số giá trị biểu diễn càng nhiều nhưng việc thực hiện tính toánc cũng như lưu trữ phức tạp  nên mất thời gian xử lý đặc biệt với các phép như nhân ma trận.

### Vậy Quantization thực hiện chuyển đổi giá trị như thế nào?

Quantization thực chất là quá trình ánh xạ các gía trị liên tục sang một tập các giá trị rời rạc hữu hạn nhỏ hơn bằng một số giải thuật xác định ô.

Xác dịnh ô là cáhc nói ta gom những giá trị gần bằng nhau ở kiểu dữ liệu ban đầu sang cùng một giá trị mới ở kiểu dữ liệu mới do bị giới hạn về số lượng giá trị có thể biểu diễn.

Ví dụ hai số 2.12234, 2.934 ở dạng float32 có thể cùng được đưa về 23 ở int8 vì cả hai đều nằm trong một ô có khoảng [2, 3].

Trong bài viết này mình sẽ không giải thích sâu về mặt toán học. Các bạn quan tâm đến phần này có thể tham khảo bài viết $\text{Quantization for Neural Networks}$. Các số sau khi thực hiện quantization sẽ có sai số so với số ban đầu do làm tròn (underflow, overflow), ...

## II. Quantization trong Pytorch

Pytorch cũng như nhiều framework khác như Tensorflow,.... đều hỗ trợ quantization. Tuy nhiên, cá nhân mình thấy việc tiếp cận và làm quen với Pytorch khá dễ dàng đồng thời cũng có khá nhiều mô hình hiện tại trên nền tảng mã nguồn mở như github do đó mình quyết định chọn sử dụng Pytorch trong bài viết này.

### 1. Ưu điểm

Pytorch hỗ trợ quantize mô hình từ float32 (mặc định của torch) về int8 nhờ đó mô hình chúng ta có thể:
- Kích thước mô hình có thể giảm tới 4 lần
- Băng thông bộ nhớ có thể giảm 4 lần
- Tốc độ xử lý có thể nhanh hơn 2 đến 4 lần với tính toán bằng float32

### 2. Quantization Mode

Pytorch cung cấp cho chúng ta hai chế độ quantization khác nhau:
- $\text{Eager Mode Quantization}$: Ở chế độ này, chúng ta cần hợp nhất các lớp như convolution, batchnorm, relu và xác định vị trí bắt đầu và kết thúc quantization thủ công. Và chúng ta chỉ sử dụng được các module thuộc torch hỗ trợ.
- $\text{FX Grapg Mode Quantization}$: Là một framework hỗ trợ quantization tự động của pytorch. Đây là một phiên bản nâng cấp của Eager Mode Quantization, hỗ trợ thêm các hàm thay vì chỉ module thuộc torch.nn như Eager Mode Quantization. Tuy nhiên chúng ta cần sửa đổi lại mô hình ban đầu để phù hợp X Graph Mode Quantization.

![](image2.png)

### 3. Giải thuật quantization

Cả hai chế độ Eager Mode Quantization và FX Graph Mode Quantization đều hỗ trợ 3 giải thuật quantization dưới đây:
- Dynamic Quantization
- Static Quantization
- Quantization-aware training

#### 3.1 Dynamic Quantization

Dynamic ở đây có nghĩa rằng việc tối ưu thuật toán quantization được diễn ra trong quá trình $\text{inference}$. Trong đó trọng số mô hình được quantize ngay lập tức còn các hàm activation sẽ được quantize vào lúc inference. $\text{Dynamic Quantization}$ thực hiện việc chuyển đổi bằng cách nhân giá trị đầu vào với một giá trị được gọi là $\text{scaling factor}$ sau đó làm tròn kết quả này tới giá trị gần nhất và lưu trữ chúng.

Dynamic Quantization là phương pháp kém hiệu quả nhất trong ba phương pháp do sự đơn giản của nó tuy nhiên phương pháp này thường được dùng trong những trường hợp thời gian thực thi bị ảnh hưởng nhiều bởi thời gian tải trọng số từ bộ nhớ hơn là do phép nhân ma trận. Bởi vậy, phương pháp này thường được sử dụng cho các mô hình như LSTM, Transformer, ...

In [ ]:
import torch

quantized_model = torch.quantization.quantize_dynamic(model, {torch.nn.Linear}, dtype=torch.qint8)

Trong đó:
- $model$: chính là model cần tối ưu
- $torch.nn.Linear$: là tập hợp các lớp trong mô hình cần quantize
- $dtype$: là kiểu dữ liệu quantize muốn chuyển về

#### 3.2 Static Quantization

Static Quantization hay còn được gọi là Post Training Quantization. So với phương pháp đầu tiên, static quantization có 4 điểm khác biệt:
- $\text{Điểm thứ nhất}$: là thực hiện quantize weights và activations của mô hình trước khi inference.
- $\text{Điểm thứ hai}$: là có thêm một bước tiến hành tinh chỉnh lại mô hình sau khi quantize, điều này đảm bảo mô hình sau khi quantize đạt độ chính xác cao hơn so với việc thực hiện lúc inference.
- $\text{Điểm thứ ba}$: là độ chính xác phụ thuộc vào phần cứng. Do Pytorch sử dụng 2 thư viện để hỗ trợ chuyển đôi là: $FBGEMM$ trên chip x86 và $QNNPACK$ trên chip ARM. Do đó cần đảm bảo máy chúng ta dùng để huấn luyện và triển khai cần giống nhau về mặt kiến trúc.
- $\text{Điểm thứ tư}$: là chúng ta cần thực hiện $\text{fuse layer}$ hay gộp các lớp convolution, batchnorm, relu thành một nhờ lớp $\text{nn.Sequential}$. Nhờ gom nhiều lớp thành một như này cho phép các thư viện tính toán trong một lần duy nhất qua đó cải thiện hiệu năng mô hình.

In [ ]:
import torch 

model = torch.quantization.fuse_modules(model, [['conv', 'bn', 'relu']])

Ví dụ cách sử dụng Static Quantization, các bạn có thể tham khảo bài viết (BETA) STATIC QUANTIZATION WITH EAGER MODE IN PYTORCH để theo dõi chi tiết hơn.

#### 3.3 Quantize Aware Training (QAT)

##### 3.3.1 QAT Hoạt động như thế nào?

$\text{QAT}$ mô hình hoá những ảnh hưởng của quantization trong suốt quá trình huấn luyện và hiệu chỉnh nó nhờ đó giúp cho phương pháp này đạt hiệu quả cao hơn so với các phương pháp quantization khác.

QAT bao gồm $4$ bước sau:
1. Huấn luyện mạng với các phép tính ở dạng dấu phẩy động.
2. Chèn các lớp $\text{Fake Quantization Q}$ vào trong mạng vừa được huấn luyện xong.
3. Thực hiện finetune mô hình. Lưu ý trong quá trình này gradient vẫn được sử dụng dưới dạng dấu phẩy động.
4. Thực hiện inference bằng cách loại bỏ $Q$ và load weight đã được quantize.

![](image3.png)

##### 3.3.2 Lớp Fake Quantization

$\text{QAT}$ hoạt động bằng cách chèn những lớp $\text{Fake Quantization}$ vào trong mô hình.

$$
Q_{fake}(x) = D(Q(x))
$$

Trong đó:
- $Q$ là một hàm quantization sẽ ánh xạ các giá trị ở số phẩy động về dạng số nguyên.
- $D$ là hàm dequantization sẽ ánh xạ ngược các giá trị đã được quantize bằng hàm $Q$ về dạng số phẩy động.

Gọi chúng là các lớp $\text{Fake Quantization}$ bởi vì chúng mô phỏng quantization bằng cách sử dụng các phép tính ở dạng số phẩy động. Và tất nhiên quá trình này diễn ra ở hai chiều $forward$ và $backward propagation$.

Ví dụ ta có một lớp $\text{Fake Quantization}$ có công thức hoạt động như sau:
$$
Q_{fake}(x,s,b) = \frac{s}{2^{b-1}}.\text{clamp}(\text{round}(\frac{2^{b-1}.x}{s}), -2^{b-1}, 2^{b-1} - 1)
$$

Trong đó:
- $x$: là input tensor ở dạng số phảy động
- $s$: là một scale factors
- $b$: là số bit để quantize
- $\text{round}$: hàm làm tròn
- $\text{clamp(x, min, max)}$: hàm giới hạn $x$ trong khoảng [min, max]

Ở đây ta nhận thấy phần tử $\frac{s}{2^{b-1}}$ đóng vai trò như hàm dequantization trong khi phần còn lại là hàm quantization. $\text{Scale Factor} \space s$ ở đây đóng vai trò là giới hạn biểu diễn của $x$ sau khi quantize. Giá trị hàm $Q$ sẽ nằm trong khoảng [$-s,s$].

Nếu $s$ nhỏ thì sẽ giới hạn dải $x$ có thể biểu diễn sau khi quantize tuy nhiên sai số sẽ nhỏ. Còn nếu $s$ lớn thì ngược lại.

Ở phần bên dưới bài viết, chúng ta cùng đi vào phần thực hành sử dụng phương pháp này với thư viện vietocr. Nhưng tạm thời chúng ta sẽ gác lại để lướt qua một vài điểm cần lưu ý khi sử dụng quantization với pytorch.

### 4. Một số lưu ý

Phần này mình có thấy bài viết A developer-friendly guide to model quantization with PyTorch khá đầy đủ, mình tham khảo và bổ sung chi tiết hơn theo ý hiểu của mình. Các bạn có thể đọc bài viết gốc bằng cách vào trực tiếp đường dẫn bên trên.

#### 4.1 Quantization chỉ là phương pháp dùng khi inference.

![](image4.png)

Như chúng ta đã biết các số dấu phẩy động có khả năng biểu diễn chính xác hơn nhiều so với các số nguyên (int8). Do đó int8 không thể dùng trong quá trình lan truyền ngược (backpropagation) vì quá trình này rất nhạy cảm với biểu diễn không chính xác của weight và dẫn tới mô hình bị phân kỳ.

#### 4.2 Độ chính xác sẽ giảm sau khi quantization?

Quantization thường làm giảm độ chính xác của mô hình. Đấy là vấn đề $tradeoff$ giữa độ chính xác và thời gian xử lý. Tuy nhiên, việc chúng ta đánh đổi bao nhiêu độ chính xác để giảm thời gian xử lý phụ thược voà rát nhiều yeus tố như kích thước mô hình ban đầu, kĩ thuật quantization hay việc húng ta quantize bao nhiêu lớp trong mô hình và lớp đó có ảnh hưởng như thế nào đến toàn bộ mô hình...

Ví dụ một mô hình có kích thước lớn thường có nhiều kết nối dư thừa hay môi hình vẫn biểu diễn tốt với ít kết nối hơn do đó quảntize sẽ không gây ảnh hưởng quá nhiều. Những yếu tố này đều được cần nghiên cứu kĩ càng để chúng ta có thể thực hiện tối ưu mô hình một cách tốt nhất.

#### 4.3 Không cần thực hiện quantization đối với toàn bộ mô hình.

Chúng ta hoàn toàn có thể quantize một phần mô hình và xác định lớp nào được quantize hay không. Để thực hiện điều này, Pytorch cung cấp cho chúng ta hai cách để thực hiện như sau:
- Tắt/bật chế độ quantization của từng lớp bằng gán giá trị $\text{.qconfig}$ của các lớp với một giá trị $qconfig_dict$ cụ thể. Ví dụ conv1.qconfig = None $\rightarrow$ nghĩa là conv1 không được quantize hoặc conv1.config = custom_qconfig $\rightarrow$ là sử dụng custom_qconfig thay cho config mà ta đã chỉ định sẫn.
- Dùng $\text{QuantStub}$ và $\text{DeQuantSub}$.

In [ ]:
import torch

# define a floating point model where some layers could be statically quantized
class M(nn.Module):
    def __init__(self, model_fp32):
        super(M, self).__init__()
        
        # QuantStub converts tensors from floating point to quantized.
        # This will only be used for inputs.
        self.quant = torch.quantization.QuantStub()
        
        # DeQuantStub converts tensors from quantized to floating point.
        # This will only be used for outputs.
        self.dequant = torch.quantization.DeQuantStub()
        
        # FP32 model
        self.model_fp32 = model_fp32

    def forward(self, x):
        # manually specify where tensors will be converted from floating
        # point to quantized in the quantized model
        x = self.quant(x)
        x = self.model_fp32(x)
        
        # manually specify where tensors will be converted from quantized
        # to floating point in the quantized model
        x = self.dequant(x)
        return x


#### 4.4 Pytorch chỉ hỗ trợ quantization với CPU

Bạn có thể vô tư thực hiện huấn luyện với Quantize Aware Training ở trên các thiết bị GPU tuy nhiên khi thực hiện inference sử dụng quantization bắt buộc bạn phải sử dụng cpu hoặc trên mobie.

### 5. Thực hành quantize mô hình VietOCR

Mọi người chắc hẳn đã quen thuộc với thư viện VietOCR - một thư viện OCR cho tiếng Việt. Ở bài trước, mình cũng đã có bài Chuyển đổi mô hình học sâu về ONNX hướng dẫn mọi người chuyển mô hình VietOCR qua dạng ONNX - một định dạng được Pytorch hỗ trợ tối ưu cũng như dễ dàng trong triển khai mô hình. Ở trong bài viết hôm nay, mình cũng sẽ giới thiệu phương pháp quantization giúp cho mô hình VietOCR chạy nhanh hơn trên những thiết bị CPU hoặc edge device. Các bạn có thể xem toàn bộ phần mã nguồn ở đây nhé. Mình cùng bắt tay vào làm nào 😃

#### 5.1 Định nghĩa cấu hình huấn luyện

Mình sẽ định nghĩa các tham số dùng cho lúc huấn luyện mô hình ở đây.

In [ ]:
config = Cfg.load_config_from_name('vgg_seq2seq')
dataset_params = {
    'name':'hw',
    'data_root':'./data_line/',
    'train_annotation':'train_line_annotation.txt',
    'valid_annotation':'test_line_annotation.txt'
}

params = {
         'print_every':200,
         'valid_every':15*200,
          'iters':20000,
          'checkpoint':'./weights/transformerocr.pth',    
          'export':'./weights/transformerocr.pth',
          'metrics': 10000
         }

config['trainer'].update(params)
config['dataset'].update(dataset_params)
config['device'] = 'cuda:1'
config['cnn']['pretrained']=False
config['weights'] = "./weights/transformerocr.pth"


#### 5.2. Chuẩn bị mô hình cho quantize aware training.

Khởi tạo mô hình và load dữ liệu từ weight có sẵn.

In [ ]:
# get pretrained model
model, vocab = build_model(config)
weights = config['weights']
model.load_state_dict(torch.load(weights, map_location=torch.device(device)))


Mô hình bên dưới sẽ giúp chúng ta quantize một phần nhỏ trong toàn bộ mô hình

In [ ]:
class QuantizedCNN(nn.Module):
    def __init__(self, model_fp32):
        super(QuantizedCNN, self).__init__()
        
        # QuantStub converts tensors from floating point to quantized.
        # This will only be used for inputs.
        self.quant = torch.quantization.QuantStub()
        
        # DeQuantStub converts tensors from quantized to floating point.
        # This will only be used for outputs.
        self.dequant = torch.quantization.DeQuantStub()
        
        # FP32 model
        self.model_fp32 = model_fp32

    def forward(self, x):
        # manually specify where tensors will be converted from floating
        # point to quantized in the quantized model
        x = self.quant(x)
        x = self.model_fp32(x)
        
        # manually specify where tensors will be converted from quantized
        # to floating point in the quantized model
        x = self.dequant(x)
        return x


Thực hiện fuse layer. Fuse layer là kỹ thuật gộp các layer riêng lẻ như Conv + Bathcnorm + Relu, Conv + Relu, Conv + BatchNorm, Linear + Relu vào một nhóm nhờ đó có thể tính toán trong một lần qua đó cải thiện hiệu suất và tăng tốc độ tính toán.

In [ ]:
model = model.train()
for m in model.cnn.model.modules():
    if type(m) == nn.Sequential:
        for n, layer in enumerate(m):
            if type(layer) == nn.Conv2d:
                torch.quantization.fuse_modules(m, [str(n), str(n + 1), str(n + 2)], inplace=True)


Trong Pytorch, quantization chỉ hỗ trợ cho một số hàm do đó phụ thuộc vào phương pháp mà mình sử dụng hoặc thiết bị backend mà chúng ta định sử dụng là cpu hay mobie nên chúng ta cần phải chọn cấu hình cho phù hợp.

In [ ]:
quantized_cnn = QuantizedCNN(model_fp32=model.cnn)
quantized_cnn.qconfig = torch.quantization.get_default_qconfig("fbgemm")

# Print quantization configurations
print(quantized_cnn.qconfig)

# the prepare() is used in post training quantization to prepares your model for the calibration step
quantized_cnn = torch.quantization.prepare_qat(quantized_cnn, inplace=True)

model.cnn = quantized_cnn


#### 5.3. Huấn luyện mô hình

In [ ]:
model.train()
model = model.to(device)
trainer = Trainer(qmodel=model, config=config, pretrained=False)
trainer.train()


Và chúng ta thu được kết quả là kích thước mô hình đã giảm từ 85MB xuống còn 29MB. Phụ thuộc vào bộ dữ liệu sử dụng huấn luyện sẽ dẫn đến kết quả khác nhau. Trong bài hướng dẫn này, mình sử dụng tạm thời bộ dữ liệu mẫu do thư viện VietOCR cung cấp.

#### 5.4. Inference

Ở bước này, chúng ta sẽ sử dụng mô hình đã được quantize để dự đoán.

In [ ]:
# define config for inference mode
config = Cfg.load_config_from_name('vgg_seq2seq')
# Pytorch support only cpu device
config['device'] = 'cpu'
config['cnn']['pretrained']=False
config['weights'] = "./weights/quantize_transformerocr.pth"

# create quantized model
qmodel, vocab = build_model(config)

## fuse layer
qmodel = model.train()
for m in qmodel.cnn.model.modules():
    if type(m) == nn.Sequential:
        for n, layer in enumerate(m):
            if type(layer) == nn.Conv2d:
                torch.quantization.fuse_modules(m, [str(n), str(n + 1), str(n + 2)], inplace=True)
  
 # prepare model for quantize aware training
quantized_cnn = QuantizedCNN(model_fp32=qmodel.cnn)
quantized_cnn.qconfig = torch.quantization.get_default_qconfig("fbgemm")

# Print quantization configurations
print(quantized_cnn.qconfig)

# the prepare() is used in post training quantization to prepares your model for the calibration step
quantized_cnn = torch.quantization.prepare_qat(quantized_cnn, inplace=True)
quantized_cnn = quantized_cnn.to(torch.device('cpu'))
qmodel.cnn = torch.quantization.convert(quantized_cnn, inplace=True)   

# create detector
detector = Predictor(config, qmodel=qmodel)


Tải bộ dữ liệu mẫu do thư viện VietOCR cung cấp

In [ ]:
# Download sample image
! gdown --id 1uMVd6EBjY4Q0G2IkU5iMOQ34X0bysm0b
! unzip  -qq -o sample.zip


Tiến hành dự đoán kết quả

In [ ]:
img = './sample/031189003299.jpeg'
img = Image.open(img)
plt.imshow(img)
s = detector.predict(img)
s


### 6. Lời kết

Đến đây nhiều bạn chắc chắn sẽ có thắc mắc tại sao mình mới quantize phần CNN còn phần encoder và decoder thì sao ? Bởi vì QAT chỉ tốt nhất cho những kiến trúc convolution. Còn đối với kiến trúc như LSTM, GRU, Transformer, chúng ta thường sử dụng phương pháp dynamic quantization. Cách này tương đối đơn giản. Các bạn có thể xem lại bài viết trước để nắm rõ thêm. Cảm ơn các bạn đã theo dõi bài viết của mình và đừng quên upvote cho mình. Nếu có bất cứ thắc mắc nào về bài viết, các bạn hãy comment xuống bên dưới để được giải đáp nhé!